# s3figure.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib as mpl
from matplotlib import font_manager
from pathlib import Path
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

# Register and verify the project-local Times New Roman fonts
times_font_dir = Path(
    configured_path('font_dir')
)
times_font_files = [
    times_font_dir / 'Times New Roman.ttf',
    times_font_dir / 'Times New Roman Bold.ttf',
]
for font_file in times_font_files:
    if not font_file.is_file():
        raise FileNotFoundError(f'Required Times New Roman font not found: {font_file}')
    font_manager.fontManager.addfont(str(font_file))
resolved_times = Path(
    font_manager.findfont('Times New Roman', fallback_to_default=False)
).resolve()
if resolved_times not in {font.resolve() for font in times_font_files}:
    raise RuntimeError(f'Times New Roman resolved to an unexpected file: {resolved_times}')

# Global figure typography
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'mathtext.cal': 'Times New Roman:italic',
    'mathtext.sf': 'Times New Roman',
    'mathtext.tt': 'Times New Roman',
})

In [ ]:
fdir1 = configured_path('reccap_mask_file')
mask = xr.open_mfdataset(fdir1)
mask_ocean = mask.open_ocean

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-1',
    periods=ds.sizes['time'],
    freq='MS'  # 按数据实际长度生成月初时间
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)
ds

In [ ]:
map_true1 = ds.MLD_true.where(mask_ocean)
map_pred1 = ds.MLD_pred.where(mask_ocean)

In [ ]:
bias = (map_pred1-map_true1)
mld_year = (
    bias
    .groupby('time.year')
    .apply(
        lambda x: x.weighted(x['time'].dt.days_in_month).mean(dim='time')
    )
)
# 2️⃣ 空间权重（cos(lat)）
lat_rad = np.deg2rad(mld_year['lat'])
weights_lat = np.cos(lat_rad)

# 3️⃣ 年平均 + 空间平均
mld_year_space_mean = (
    mld_year
    .weighted(weights_lat)
    .mean(dim=('lat', 'lon'))
)

# STE：每年 12 个月的全部有效格点，std / sqrt(n)
annual_bias_std = (
    bias
    .groupby('time.year')
    .std(dim=('time', 'lat', 'lon'), skipna=True)
)
annual_bias_n = (
    bias.notnull()
    .groupby('time.year')
    .sum(dim=('time', 'lat', 'lon'))
)
annual_month_count = bias['time'].groupby('time.year').count()
mld_year_space_ste = (annual_bias_std / np.sqrt(annual_bias_n)).where(
    annual_month_count == 12
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# =========================
# 1. 数据准备
# =========================
yearly_valid_sample_count = (
    map_true1
    .notnull()
    .sum(dim=('lat', 'lon'))   # 先变成 (time,)
    .groupby('time.year')
    .sum(dim='time')           # 再按年加
)
years = yearly_valid_sample_count.coords['year'].values
count_values = yearly_valid_sample_count.values
mld_values = mld_year_space_mean.sel(year=years).values
mld_ste_values = mld_year_space_ste.sel(year=years).values

# =========================
# 2. 画图
# =========================
fig, ax1 = plt.subplots(figsize=(8, 4))

# ---- 左轴：年平均 MLD ----
line1 = ax1.plot(
    years,
    mld_values,
    color='tab:blue',
    linewidth=2,
    marker='s',
    markersize=4
)
shading = ax1.fill_between(
    years,
    mld_values - mld_ste_values,
    mld_values + mld_ste_values,
    color='tab:blue',
    alpha=0.20,
    linewidth=0,
    zorder=1
)

ax1.axhline(
    0,
    color='black',
    linewidth=1,
    linestyle='--',
    zorder=0
)

ax1.set_xlabel('Date (YYYY)', fontsize=18)
# 每 5 年显示一次刻度：1985、1990、1995……
ax1.xaxis.set_major_locator(
    mticker.FixedLocator(np.arange(1985, years.max() + 1, 5))
)
ax1.set_ylabel('Bias (m)', fontsize=18, color='tab:blue')
ax1.tick_params(axis='x', labelsize=16)
ax1.tick_params(axis='y', labelsize=16, labelcolor='tab:blue')

ax1.grid(True, linestyle='--', alpha=0.5)

# ---- 右轴：样本数 ----
ax2 = ax1.twinx()

line2 = ax2.plot(
    years,
    count_values,
    color='green',
    linewidth=2,
    marker='o',
    markersize=4
)

ax2.set_ylabel('Data count', fontsize=18, color='green')
ax2.tick_params(axis='y', labelsize=16, labelcolor='green')

# 科学计数法（只对样本数）
ax2.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax2.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax2.yaxis.get_offset_text().set_fontsize(16)

# 对齐双 y 轴：左轴 0 ↔ 右轴 6×10^4；左轴 -0.5 ↔ 右轴 4×10^4
left_ymin, left_ymax = ax1.get_ylim()
right_per_left = 4.0e4
right_at_left_zero = 6.0e4
ax2.set_ylim(
    right_at_left_zero + right_per_left * left_ymin,
    right_at_left_zero + right_per_left * left_ymax,
)

plt.tight_layout()
output_dir = Path(
    configured_path('output_dir')
)
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(
    output_dir / 's3figure.png',
    dpi=600,
    bbox_inches='tight',
    pad_inches=0,
)
plt.show()
